# Harvest essentiality labels from PubMed with BiomedBERT

Mine published essentiality / Tn-seq statements into candidate (organism, gene, essential/non-essential, condition) labels to feed the predictor. The bottleneck of the whole project is LABELS, not architecture -- this targets that.

**Honest design note**: `microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext` is a pretrained ENCODER (masked-LM), not a ready NER/relation extractor. We use it correctly two ways:
  1. **zero-shot cloze scorer** (native masked-LM ability) -- score whether a gene is asserted essential vs dispensable, no fine-tuning
  2. entity tagging via a fine-tuned biomedical NER checkpoint (BiomedBERT base would need fine-tuning first)

Output = candidate triples WITH provenance (PMID + sentence) for human curation. Text-mining is noisy; this produces a curation queue, not gold labels.

**Highest-value target**: CONDITIONAL essentiality statements ('X required for growth under Y') -- the rogue-zone signal that exists ONLY in free text, in no database.

GPU recommended (BiomedBERT inference). PubMed is reachable from Colab.

## 0. FIRST: audit existing databases (maybe the labels are already free)

In [ ]:
# before mining text, check what DEG already has that our labels.csv lacks.
!git clone --depth 1 -b claude/vectorize-gex-propensity-NRqBW https://github.com/nikku03/cell.git cell_repo || echo cloned
import os, csv; os.chdir('cell_repo')
ours = set(r['organism'] for r in csv.DictReader(open('data/drive_import/labels/labels.csv')))
print(f'our labelled organisms: {len(ours)}')
# DEG bulk already in repo?
import glob; print('DEG files in repo:', glob.glob('data/drive_import/deg/*'))
# inspect what organisms DEG covers vs ours
try:
    deg = open('data/drive_import/deg/deg_bacteria.csv').read()[:500]; print(deg)
except Exception as e: print('check DEG path:', e)

## 1. Install

In [ ]:
!pip install -q transformers torch requests
import torch; print('cuda', torch.cuda.is_available())

## 2. Fetch essentiality / Tn-seq papers from PubMed (E-utilities)

In [ ]:
import requests, time, re
EUTILS='https://eutils.ncbi.nlm.nih.gov/entrez/eutils'
QUERY='(gene essentiality OR transposon sequencing OR Tn-seq OR essential genes) AND bacteria AND ("2015"[Date - Publication] : "3000"[Date - Publication])'
r=requests.get(f'{EUTILS}/esearch.fcgi', params=dict(db='pubmed', term=QUERY, retmax=400, retmode='json'), timeout=60)
ids=r.json()['esearchresult']['idlist']
print(f'{len(ids)} PMIDs')
# fetch abstracts in batches
abstracts={}
for i in range(0,len(ids),100):
    batch=ids[i:i+100]
    ef=requests.get(f'{EUTILS}/efetch.fcgi', params=dict(db='pubmed', id=','.join(batch), rettype='abstract', retmode='xml'), timeout=120)
    for m in re.finditer(r'<PMID[^>]*>(\d+)</PMID>.*?<AbstractText[^>]*>(.*?)</AbstractText>', ef.text, re.S):
        pmid, txt = m.group(1), re.sub(r'<[^>]+>','',m.group(2))
        abstracts.setdefault(pmid, txt)
    time.sleep(0.4)
print(f'{len(abstracts)} abstracts with text')

## 3. Split into sentences; keep those asserting essentiality

In [ ]:
ESS_POS=r'\bessential\b|required for (growth|viability|survival)|indispensable|could not be (deleted|disrupted|inactivated)|lethal'
ESS_NEG=r'\b(non-?essential|dispensable)\b|not (required|essential)|tolerat(ed|es) (deletion|disruption)'
COND=r'under (.*?)(stress|condition|medium|presence of|exposure)|during (infection|biofilm|starvation)|in the presence of (\w+)'
sents=[]
for pmid,txt in abstracts.items():
    for s in re.split(r'(?<=[.!?])\s+', txt):
        if re.search(ESS_POS, s, re.I) or re.search(ESS_NEG, s, re.I):
            sents.append((pmid, s.strip()))
print(f'{len(sents)} candidate essentiality sentences')
for pmid,s in sents[:5]: print(f'  [{pmid}] {s[:160]}')

## 4. NER: tag genes + organisms (fine-tuned biomedical NER)
Base BiomedBERT has no NER head, so use a ready fine-tuned biomedical NER. Swap MODEL if you fine-tune BiomedBERT yourself.

In [ ]:
from transformers import pipeline
# d4data biomedical NER covers many bio entity types incl. genes/proteins/organisms
ner = pipeline('token-classification', model='d4data/biomedical-ner-all',
               aggregation_strategy='simple', device=0 if __import__('torch').cuda.is_available() else -1)
# also a lightweight gene-token heuristic for bacterial gene names (e.g. ftsZ, dnaA, lpxC)
GENE_RE=re.compile(r'\b[a-z]{3}[A-Z][0-9]?\b')   # 3 lowercase + 1 uppercase = classic bacterial gene symbol
def entities(sentence):
    genes=set(GENE_RE.findall(sentence)); orgs=set()
    for e in ner(sentence):
        grp=e['entity_group'].lower()
        if 'gene' in grp or 'protein' in grp: genes.add(e['word'])
        if 'organism' in grp or 'species' in grp or 'bacteri' in grp: orgs.add(e['word'])
    return genes, orgs
g,o = entities(sents[0][1]); print('sample genes', g, '| orgs', o)

## 5. BiomedBERT zero-shot cloze: confirm the asserted polarity
Native masked-LM use: probe whether BiomedBERT itself scores 'essential' > 'dispensable' for the gene-in-context. A confidence signal on top of the rule-based polarity.

In [ ]:
from transformers import AutoTokenizer, AutoModelForMaskedLM
import torch
tok=AutoTokenizer.from_pretrained('microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext')
mlm=AutoModelForMaskedLM.from_pretrained('microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext')
mlm.eval();
if torch.cuda.is_available(): mlm=mlm.cuda()
ESS_ID=tok.convert_tokens_to_ids('essential'); DISP_ID=tok.convert_tokens_to_ids('dispensable')
def cloze_essential(gene, organism='bacteria'):
    text=f'In {organism}, the {gene} gene is [MASK] for growth.'
    enc=tok(text, return_tensors='pt')
    if torch.cuda.is_available(): enc={k:v.cuda() for k,v in enc.items()}
    with torch.no_grad(): logits=mlm(**enc).logits
    mpos=(enc['input_ids']==tok.mask_token_id).nonzero()[0,1]
    probs=torch.softmax(logits[0,mpos],-1)
    pe,pd=float(probs[ESS_ID]),float(probs[DISP_ID])
    return pe/(pe+pd+1e-9)   # ->1 essential-leaning, ->0 dispensable
print('cloze(ftsZ):', round(cloze_essential('ftsZ'),3), '| cloze(lacZ):', round(cloze_essential('lacZ'),3))

## 6. Assemble candidate triples with provenance

In [ ]:
import pandas as pd
cand=[]
for pmid, s in sents:
    neg=bool(re.search(ESS_NEG, s, re.I)); pos=bool(re.search(ESS_POS, s, re.I))
    label='non-essential' if (neg and not pos) else 'essential' if pos else 'ambiguous'
    cond=re.search(COND, s, re.I); condition=cond.group(0)[:60] if cond else 'unspecified'
    genes, orgs = entities(s)
    for gene in list(genes)[:5]:
        org = list(orgs)[0] if orgs else 'unspecified'
        cl = cloze_essential(gene, org if org!='unspecified' else 'bacteria')
        cand.append(dict(pmid=pmid, gene=gene, organism=org, label=label,
                         condition=condition, cloze_essential=round(cl,3), sentence=s[:300]))
cdf=pd.DataFrame(cand).drop_duplicates(['pmid','gene','organism'])
print(f'{len(cdf)} candidate triples')
# agreement filter: rule-label and cloze agree -> higher confidence
cdf['agree']=((cdf.label=='essential')&(cdf.cloze_essential>0.6)) | ((cdf.label=='non-essential')&(cdf.cloze_essential<0.4))
print(f'high-confidence (rule + cloze agree): {cdf.agree.sum()}')
cdf.to_csv('outputs/orphan/pubmed_candidate_labels.csv', index=False)
print(cdf[cdf.agree].head(12)[['pmid','organism','gene','label','condition','cloze_essential']].to_string())

## 7. Audit: which are NEW conditional statements vs what databases have

In [ ]:
cond_stmts=cdf[(cdf.condition!='unspecified')&cdf.agree]
print(f'CONDITIONAL essentiality statements (the rogue-zone signal, not in databases): {len(cond_stmts)}')
print(cond_stmts.head(15)[['organism','gene','label','condition']].to_string())
print('\nThese are the highest-value rows -- conditional essentiality exists only in free text.')
print('Next: normalize gene+organism -> locus_tag/OG, curate, append to labels.csv, retrain.')

## Reality check (read before trusting any of this)

- **Output is a CURATION QUEUE, not gold labels.** Expect ~30-50% of rows to be wrong (gene/organism mis-tagged, polarity flipped by negation, homolog references like 'the E. coli ftsZ homolog'). The `agree` flag (rule + cloze) is a first filter, not a guarantee.
- **Normalization is the hard, unfinished step.** Gene symbol -> locus_tag -> OG requires NCBI Gene + our orthology, and it is lossy. Until that's done these can't enter the model.
- **Binary essentiality is probably already in DEG** (cell 0). The unique value here is the CONDITIONAL statements (cell 7) -- those exist in no database.
- **BiomedBERT base did exactly two honest jobs**: zero-shot cloze polarity scoring (cell 5) and (if you fine-tune it) NER. It is NOT predicting essentiality -- it is extracting what humans already measured and wrote down.
- To make this production-grade: fine-tune BiomedBERT on a small hand-labeled set of essentiality sentences for relation extraction; add a proper gene-normalization step (e.g., via NCBI Gene esearch per (gene, organism)).